In [1]:

import sys
import numpy as np
import pandas as pd 
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import networkx as nx
from sklearn.preprocessing import StandardScaler
np.set_printoptions(threshold=sys.maxsize)
import warnings
warnings.filterwarnings("ignore", category=Warning)


# Useful functions 

In [2]:

def filter_label(df, y, label):
    # Splits the dataset into two parts based on the specified label.
    # Returns the subset of the dataframe and labels matching the label,
    # and the subset of the dataframe and labels not matching the label.
     
    output_df = df[np.where(y == label)[0]]
    
    res_df =  df[np.where(y != label)[0]]
    
    output_y = y[np.where(y == label)[0]]
    res_y = y[np.where(y != label)[0]]
    
    return  output_df, output_y, res_df, res_y

 


def obtain_labels(df, label_path):
    # Maps feature data from the dataframe to their corresponding labels from a CSV file.
    # Returns arrays of features and their corresponding labels.
    
    labels = pd.read_csv(label_path, header=0)
    
    y = []
    x = []
    
    for index, row in df.iterrows():
        #print(row["asm_id"].split(".")[0])
        hash_id = row["asm_id"].split(".")[0]
        if hash_id in labels['asm_id'].values: 
            row = row.drop("asm_id")
            x.append(row)
            y.append(labels[labels["asm_id"] == hash_id]["Class"])
            

    
    return np.array(x), np.array(y)


def change_attack_label(x):
    # Changes the attack label to 1.
    # Returns a list with a single element [1.].
    label = [1.]
    return label




# A utility method to create a tf.data dataset from a Pandas Dataframe
def df_to_dataset(x, y, shuffle=True, batch_size=128, epochs = 60):
    ds = tf.data.Dataset.from_tensor_slices((x, y)).repeat(epochs)
    #ds = tf.data.Dataset.from_tensor_slices((dict(dataframe), labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(x))
    ds = ds.batch(batch_size)
    return ds



# Load malware data

In [3]:
path = "data/content_features/Big15_2/FeatureCategories/train/asm_full.csv"
malware_x = pd.read_csv(path, header=0)
print(malware_x.shape)

(10864, 965)


In [4]:
label_path = "data/labels/new_gmm_labels_id.csv"

malware_x, malware_y = obtain_labels(malware_x, label_path)
print(malware_x.shape)
print(malware_y.shape)

(10708, 964)
(10708, 1)


In [5]:
#scale the data using StandardScaler
malware_x= StandardScaler().fit_transform(malware_x)

# Load benign data source

In [6]:
path = "data/content_features/benign_source/dataset1/asm_full.csv"
source_normal_x_1 = pd.read_csv(path, header=0)
source_normal_x_1 = source_normal_x_1.drop("id", axis = 1)
source_normal_y_1 = np.zeros((source_normal_x_1.shape[0],1))


path = "data/content_features/benign_source/dataset2/asm_full.csv"
source_normal_x_2 = pd.read_csv(path, header=0)
source_normal_x_2 = source_normal_x_2.drop("asm_id", axis = 1)
source_normal_y_2 = np.zeros((source_normal_x_2.shape[0],1))


path = "data/content_features/benign_source/dataset3/asm_full.csv"
source_normal_x_3 = pd.read_csv(path, header=0)
source_normal_x_3 = source_normal_x_3.drop("asm_id", axis = 1)
source_normal_y_3 = np.zeros((source_normal_x_3.shape[0],1))


path = "data/content_features/benign_source/dataset4/asm_full.csv"
source_normal_x_4 = pd.read_csv(path, header=0)
source_normal_x_4 = source_normal_x_4.drop("asm_id", axis = 1)
source_normal_y_4 = np.zeros((source_normal_x_4.shape[0],1))


source_normal_x = np.concatenate((source_normal_x_1, source_normal_x_2, source_normal_x_3, source_normal_x_4),axis =0)
source_normal_y = np.concatenate((source_normal_y_1, source_normal_y_2, source_normal_y_3, source_normal_y_4),axis =0)
print(source_normal_x.shape)
print(source_normal_y.shape)

source_normal_x= StandardScaler().fit_transform(source_normal_x)

(7665, 964)
(7665, 1)


# Load benign data target

In [7]:
path = "data/content_features/benign_target/dataset1/asm_full.csv"
target_normal_x_1 = pd.read_csv(path, header=0)
target_normal_x_1 = target_normal_x_1.drop("asm_id", axis = 1)
target_normal_y_1 = np.zeros((target_normal_x_1.shape[0],1))


path = "data/content_features/benign_target/dataset2/asm_full.csv"
target_normal_x_2 = pd.read_csv(path, header=0)
target_normal_x_2 = target_normal_x_2.drop("asm_id", axis = 1)
target_normal_y_2 = np.zeros((target_normal_x_2.shape[0],1))


path = "data/content_features/benign_target/dataset3/asm_full.csv"
target_normal_x_3 = pd.read_csv(path, header=0)
target_normal_x_3 = target_normal_x_3.drop("asm_id", axis = 1)
target_normal_y_3 = np.zeros((target_normal_x_3.shape[0],1))


path = "data/content_features/benign_target/dataset4/asm_full.csv"
target_normal_x_4 = pd.read_csv(path, header=0)
target_normal_x_4 = target_normal_x_4.drop("asm_id", axis = 1)
target_normal_y_4 = np.zeros((target_normal_x_4.shape[0],1))


target_normal_x = np.concatenate((target_normal_x_1, target_normal_x_2, target_normal_x_3, target_normal_x_4),axis =0)
target_normal_y = np.concatenate((target_normal_y_1, target_normal_y_2, target_normal_y_3, target_normal_y_4),axis =0)
print(target_normal_x.shape)
print(target_normal_y.shape)

target_normal_x= StandardScaler().fit_transform(target_normal_x)

(7257, 964)
(7257, 1)


# Build model

In [8]:
class DANN_NN(object):
    def __init__(self, x_source_train, y_source_train, 
                 x_target_train, y_target_train, 
                 x_source_test, y_source_test, 
                 x_target_test, y_target_test,  nSteps=20000):

        # Source train and test dataset
        self.x_source_train = x_source_train
        self.y_source_train = y_source_train
        
        self.x_source_test = x_source_test
        self.y_source_test = y_source_test
    
        # Target train and test dataset
        self.x_target_train = x_target_train
        self.y_target_train = y_target_train
        

        self.x_target_test = x_target_test
        self.y_target_test = y_target_test
        
     
        
        self.n_classes = y_source_train.shape[1]
        

                
        # Use the source dataset shape for the generator input and outputs.
        self.input_shape = x_source_train.shape[1]
        self.output_shape = x_source_train.shape[1]
        
        #Latent dim for AE/VAE
        self.latent_dim = 100
        
        self.optimizer_G = Adam(0.0002,0.5)
        self.optimizer_D = Adam(0.0002,0.5) 
        self.optimizer_C = Adam(0.0002,0.5) 
         
        self.batch_size = 128
        self.nStep = nSteps


    
    def build_generator(self):
        print("\n== Build Generator...")
        
        inputs = Input(self.input_shape)
        net = Dense(units=100, activation=tf.nn.relu, name="fc_G1")(inputs)
        net = Dense(units=100, activation=tf.nn.relu, name="fc_G2")(net)        
        net = Dense(units=100, activation=tf.nn.relu, name="fc_G3")(net)
        net = Dense(units=100, activation=tf.nn.relu, name="fc_G4")(net)

        DIrep = Dense(units=self.latent_dim, activation=tf.nn.relu, name="DIrep")(net)
        G = Model(inputs=inputs, outputs=DIrep, name="Generator")
        
        inputs = Input(shape=(self.latent_dim))
        net = Dense(units=400, activation=tf.nn.relu, name="fc_C1")(inputs)
        net = Dense(units=400, activation=tf.nn.relu, name="fc_C2")(net)
        net = Dense(units=self.n_classes, activation=tf.nn.softmax, name="C")(net)
        C = Model(inputs=inputs, outputs=net, name="Classifier")
        
        return G, C

    def build_disciminator(self):
        print("\n== Build Discriminator...")
        
        inputs = Input(self.latent_dim)
        net = Dense(units=100, activation=tf.nn.relu, name="fc_D1")(inputs)
        net = Dense(units=100, activation=tf.nn.relu, name="fc_D2")(net)
        net = Dense(units=100, activation=tf.nn.relu, name="fc_D3")(net)
        net = Dense(units=100, activation=tf.nn.relu, name="fc_D4")(net)

        net = Dense(units=2, activation=tf.nn.softmax, name="D")(net)
        D = Model(inputs=inputs, outputs=net, name="Discriminator")
        return D


    def d_loss(self, yhat_source, yhat_target): 

        y_source = np.tile([1,0], (yhat_source.shape[0], 1))
        y_target = np.tile([0,1], (yhat_target.shape[0], 1))
                          
        bce = CategoricalCrossentropy(from_logits=False)        
        return bce(y_source, yhat_source) + bce(y_target, yhat_target)

    def g_loss(self, yhat_source, yhat_target):

        y_source = np.tile([0,1], (yhat_source.shape[0], 1))
        y_target = np.tile([1,0], (yhat_target.shape[0], 1))
                          
        bce = CategoricalCrossentropy(from_logits=False)        
        return bce(y_source, yhat_source) + bce(y_target, yhat_target)

    def c_loss(self, yhat_class_source, yhat_class_target, y_source, y_target):
        
        bce = CategoricalCrossentropy(from_logits=False)
            
        
        return bce(y_source, yhat_class_source) + bce(y_target, yhat_class_target)*0.1
        

    def train(self):
        D = self.build_disciminator()
        G, C = self.build_generator()
        
        

        
        # Create batch generators for source, target unlabeled and target labeled samples
        S_batches = tf.data.Dataset.from_tensor_slices((self.x_source_train, self.y_source_train)).repeat().batch(self.batch_size).as_numpy_iterator()
        T_batches = tf.data.Dataset.from_tensor_slices((self.x_target_train, self.y_target_train)).repeat().batch(self.batch_size).as_numpy_iterator()
      
        
        #optimizer = self.optimizer

        #g_loss_weight = 1
        c_loss_weight = 1
        g_loss_weight = 0.1
        

        print('====Loss Weights====')
        print('g_loss_weight: {0}'.format(g_loss_weight))
        print('c_loss_weight: {0}'.format(c_loss_weight))
        

        


        def _train_step():
            
           

            # Get a batch of source and target unlabeled samples
            x_batch_source, y_batch_source = next(S_batches)
            x_batch_target, y_batch_target = next(T_batches)
            
                 
            #Create domain invariant mapping using the Generator
            DIrep_source_samples = G(x_batch_source)
            DIrep_target_samples = G(x_batch_target)

           
            # Calculate the Domain loss
            with tf.GradientTape(persistent=True) as tape_disc:
                
                #Predict the domain using the discriminator
                yhat_source = D(DIrep_source_samples)
                yhat_target = D(DIrep_target_samples)
                

                # Compute D loss
                d_loss_value = self.d_loss(yhat_source, yhat_target)
                
            # Given loss, compute and apply gradient for discriminator:
            d_gradients = tape_disc.gradient(d_loss_value, D.trainable_variables)
            self.optimizer_D.apply_gradients(zip(d_gradients, D.trainable_variables))
            
            ########################################################################
            ########################################################################
            ########################################################################
            
            # Get a batch of source and target unlabeled samples
            x_batch_source, y_batch_source = next(S_batches)
            x_batch_target, y_batch_target = next(T_batches)
            


            with tf.GradientTape(persistent=True) as tape_gen:
                
    
                #Create domain invariant mapping using the Generator
                DIrep_source_samples = G(x_batch_source)
                DIrep_target_samples = G(x_batch_target)
                
                
                #Predict the domain using the discriminator
                yhat_source = D(DIrep_source_samples)
                yhat_target = D(DIrep_target_samples)
                

                #Predict the class of the samples
                class_pred_source = C(G(x_batch_source))
                
                
                class_pred_target = C(G(x_batch_target))
                
            
                # Compute G loss
                g_loss_value = self.g_loss(yhat_source, yhat_target)
                
                # Compute C loss
                c_loss_value = self.c_loss(class_pred_source, class_pred_target, 
                                          y_batch_source, y_batch_target)
                
                #combined_loss_value = (g_loss_weight * g_loss_value + c_loss_weight * c_loss_value) / (g_loss_weight+ c_loss_weight)

                combined_loss_value = (g_loss_weight * g_loss_value 
                                       + c_loss_weight * c_loss_value) / (g_loss_weight + c_loss_weight)
                
            # Given loss, compute and apply gradient:            
            g_gradients = tape_gen.gradient(combined_loss_value, G.trainable_variables)
            c_gradients = tape_gen.gradient(c_loss_value, C.trainable_variables)
            
    
            self.optimizer_G.apply_gradients(zip(g_gradients, G.trainable_variables))
            self.optimizer_C.apply_gradients(zip(c_gradients, C.trainable_variables))
            
            return G, C, D, g_loss_value, c_loss_value, d_loss_value

        # Start training nStep:
        for step in range(self.nStep):
            generator, classifier, discriminator,  \
            g_loss_value, c_loss_value, d_loss_value = _train_step()
            
            if step % 1000 == 0:
                y_source_pred_test = classifier.predict(generator(self.x_source_test)).argmax(1)
                y_target_pred_test = classifier.predict(generator(self.x_target_test)).argmax(1)
                
                accuracy_source = accuracy_score(self.y_source_test.argmax(1), y_source_pred_test)
                accuracy_target = accuracy_score(self.y_target_test.argmax(1), y_target_pred_test)
                f1_target = f1_score(self.y_target_test.argmax(1), y_target_pred_test)

               
                track_loss = ('Step {} ==> G_Loss: {} C_Loss: {} D_Loss: {}  \n'
                              'Acc Source: {} Acc Target: {} f1 Target: {} \n' ).format(step, g_loss_value.numpy(), c_loss_value.numpy(), 
                                                                        d_loss_value.numpy(), accuracy_source, accuracy_target, f1_target)
                                                                                    
                print(track_loss)
                
                
                
                
                
        print('Training ended')



     


# Training

## Set Cluster 0 as the target domain 

In [9]:

target_malware_x, target_malware_y, source_malware_x, source_malware_y =  filter_label(malware_x, malware_y, [0])
print("Malware data ...")
print("Target: {}".format(target_malware_x.shape))
print("Target; {}".format(target_malware_y.shape))
print("Source {}".format(source_malware_x.shape))
print("Source {}".format(source_malware_y.shape))

print("Normal data ...")
print("Target: {}".format(target_normal_x.shape))
print("Target: {}".format(target_normal_y.shape))
print("Source {}".format(source_normal_x.shape))
print("Source {}".format(source_normal_y.shape))

#concatenate source and target
source_malware_y = np.apply_along_axis(change_attack_label, 1, source_malware_y)
target_malware_y = np.apply_along_axis(change_attack_label, 1, target_malware_y)


source_x = np.concatenate((source_malware_x, source_normal_x), axis = 0)
source_y = np.concatenate((source_malware_y, source_normal_y), axis = 0)


target_x = np.concatenate((target_malware_x, target_normal_x), axis = 0)
target_y = np.concatenate((target_malware_y, target_normal_y), axis = 0)


#one-hot encode labels

source_y = tf.keras.utils.to_categorical(source_y, num_classes = 2)
target_y = tf.keras.utils.to_categorical(target_y, num_classes = 2)




print("Combined data ...")
print("Target: {}".format(target_x.shape))
print("Target: {}".format(target_y.shape))
print("Source {}".format(source_x.shape))
print("Source {}".format(source_y.shape))




# Split the data into train and test sets
source_x_train, source_x_test, source_y_train, source_y_test = train_test_split(source_x, source_y, test_size=0.25, random_state=42)
target_x_train, target_x_test, target_y_train, target_y_test = train_test_split(target_x, target_y, test_size=0.5, random_state=42)
        
    

print("train test data ...")
print("Target train: {}".format(target_x_train.shape))
print("Target train: {}".format(target_y_train.shape))
print("Target test: {}".format(target_x_test.shape))
print("Target test: {}".format(target_y_test.shape))
print("Source train: {}".format(source_x_train.shape))
print("Source train: {}".format(source_y_train.shape))
print("Source test: {}".format(source_x_test.shape))
print("Source test: {}".format(source_y_test.shape))
           

Malware data ...
Target: (6023, 964)
Target; (6023, 1)
Source (4685, 964)
Source (4685, 1)
Normal data ...
Target: (7257, 964)
Target: (7257, 1)
Source (7665, 964)
Source (7665, 1)
Combined data ...
Target: (13280, 964)
Target: (13280, 2)
Source (12350, 964)
Source (12350, 2)
train test data ...
Target train: (6640, 964)
Target train: (6640, 2)
Target test: (6640, 964)
Target test: (6640, 2)
Source train: (9262, 964)
Source train: (9262, 2)
Source test: (3088, 964)
Source test: (3088, 2)


In [11]:
samples = [20,50,100,200,300,500]

for size in samples: 
    print("---------------------{} labeled samples per class ---------------------------".format(size))
    # Select a random sample of the target data of different sizes
    idxs = np.random.permutation(target_x_train.shape[0]) 
    split = int(size)
    idx_sample, _= np.split(idxs, [split])
    target_x_train_select =  target_x_train[idx_sample]
    target_y_train_select =  target_y_train[idx_sample]
    
    print("target_x_train_select: {}".format(target_x_train_select.shape))
    print("target_y_train_select: {}".format(target_y_train_select.shape))
    print('source_x_train: {0}'.format(source_x_train.shape))
    print('source_y_train: {0}'.format(source_y_train.shape))

    # We have limited the number of training steps to 2000 to minimize training time.
    # You can change the epochs to a lower number (lowest 1) just to test if the code is functional.
    dann_nn = DANN_NN(source_x_train, source_y_train, 
                  target_x_train_select,  target_y_train_select , 
                  source_x_test, source_y_test, 
                  target_x_test,target_y_test, nSteps=2000)
    
    

    dann_nn.train()

    
    

---------------------20 labeled samples per class ---------------------------
target_x_train_select: (20, 964)
target_y_train_select: (20, 2)
source_x_train: (9262, 964)
source_y_train: (9262, 2)

== Build Discriminator...

== Build Generator...
====Loss Weights====
g_loss_weight: 0.1
c_loss_weight: 1
208/208 [==============================] - 0s 794us/step
Step 0 ==> G_Loss: 1.38889741897583 C_Loss: 0.7665008902549744 D_Loss: 1.3867568969726562  
Acc Source: 0.6172279792746114 Acc Target: 0.5430722891566265 f1 Target: 0.016212710765239946 

208/208 [==============================] - 0s 795us/step
Step 1000 ==> G_Loss: 1.4614423513412476 C_Loss: 0.0005743911024183035 D_Loss: 1.3692848682403564  
Acc Source: 0.9880181347150259 Acc Target: 0.9433734939759036 f1 Target: 0.941010354565422 

Training ended
---------------------50 labeled samples per class ---------------------------
target_x_train_select: (50, 964)
target_y_train_select: (50, 2)
source_x_train: (9262, 964)
source_y_train: (

## Set Cluster 1 as the target domain

In [12]:

target_malware_x, target_malware_y, source_malware_x, source_malware_y =  filter_label(malware_x, malware_y, [1])
print("Malware data ...")
print("Target: {}".format(target_malware_x.shape))
print("Target; {}".format(target_malware_y.shape))
print("Source {}".format(source_malware_x.shape))
print("Source {}".format(source_malware_y.shape))

print("Normal data ...")
print("Target: {}".format(target_normal_x.shape))
print("Target: {}".format(target_normal_y.shape))
print("Source {}".format(source_normal_x.shape))
print("Source {}".format(source_normal_y.shape))

#concatenate source and target
source_malware_y = np.apply_along_axis(change_attack_label, 1, source_malware_y)
target_malware_y = np.apply_along_axis(change_attack_label, 1, target_malware_y)



source_x = np.concatenate((source_malware_x, source_normal_x), axis = 0)
source_y = np.concatenate((source_malware_y, source_normal_y), axis = 0)


target_x = np.concatenate((target_malware_x, target_normal_x), axis = 0)
target_y = np.concatenate((target_malware_y, target_normal_y), axis = 0)


#one-hot encode labels

source_y = tf.keras.utils.to_categorical(source_y, num_classes = 2)
target_y = tf.keras.utils.to_categorical(target_y, num_classes = 2)




print("Combined data ...")
print("Target: {}".format(target_x.shape))
print("Target: {}".format(target_y.shape))
print("Source {}".format(source_x.shape))
print("Source {}".format(source_y.shape))




# Split the data into train and test sets
source_x_train, source_x_test, source_y_train, source_y_test = train_test_split(source_x, source_y, test_size=0.25, random_state=42)
target_x_train, target_x_test, target_y_train, target_y_test = train_test_split(target_x, target_y, test_size=0.5, random_state=42)
        
    

print("train test data ...")
print("Target train: {}".format(target_x_train.shape))
print("Target train: {}".format(target_y_train.shape))
print("Target test: {}".format(target_x_test.shape))
print("Target test: {}".format(target_y_test.shape))
print("Source train: {}".format(source_x_train.shape))
print("Source train: {}".format(source_y_train.shape))
print("Source test: {}".format(source_x_test.shape))
print("Source test: {}".format(source_y_test.shape))
           

Malware data ...
Target: (3203, 964)
Target; (3203, 1)
Source (7505, 964)
Source (7505, 1)
Normal data ...
Target: (7257, 964)
Target: (7257, 1)
Source (7665, 964)
Source (7665, 1)
Combined data ...
Target: (10460, 964)
Target: (10460, 2)
Source (15170, 964)
Source (15170, 2)
train test data ...
Target train: (5230, 964)
Target train: (5230, 2)
Target test: (5230, 964)
Target test: (5230, 2)
Source train: (11377, 964)
Source train: (11377, 2)
Source test: (3793, 964)
Source test: (3793, 2)


In [13]:
samples = [20,50,100,200,300,500]

for size in samples: 
    print("---------------------{} labeled samples per class ---------------------------".format(size))
    # Select a random sample of the target data of different sizes
    idxs = np.random.permutation(target_x_train.shape[0]) 
    split = int(size)
    idx_sample, _= np.split(idxs, [split])
    target_x_train_select =  target_x_train[idx_sample]
    target_y_train_select =  target_y_train[idx_sample]
    
    print("target_x_train_select: {}".format(target_x_train_select.shape))
    print("target_y_train_select: {}".format(target_y_train_select.shape))
    print('source_x_train: {0}'.format(source_x_train.shape))
    print('source_y_train: {0}'.format(source_y_train.shape))


    # We have limited the number of training steps to 2000 to minimize training time.
    # You can change the epochs to a lower number (lowest 1) just to test if the code is functional.
    dann_nn = DANN_NN(source_x_train, source_y_train, 
                  target_x_train_select,  target_y_train_select , 
                  source_x_test, source_y_test, 
                  target_x_test,target_y_test, nSteps=2000)
    

    dann_nn.train()

    
    

---------------------20 labeled samples per class ---------------------------
target_x_train_select: (20, 964)
target_y_train_select: (20, 2)
source_x_train: (11377, 964)
source_y_train: (11377, 2)

== Build Discriminator...

== Build Generator...
====Loss Weights====
g_loss_weight: 0.1
c_loss_weight: 1
164/164 [==============================] - 0s 764us/step
Step 0 ==> G_Loss: 1.385398507118225 C_Loss: 0.7578232288360596 D_Loss: 1.4074575901031494  
Acc Source: 0.6643817558660691 Acc Target: 0.6520076481835564 f1 Target: 0.4602609727164888 

164/164 [==============================] - 0s 808us/step
Step 1000 ==> G_Loss: 1.5083019733428955 C_Loss: 0.0007394120912067592 D_Loss: 1.3402451276779175  
Acc Source: 0.9912997627208014 Acc Target: 0.9256214149139579 f1 Target: 0.8941496598639457 

Training ended
---------------------50 labeled samples per class ---------------------------
target_x_train_select: (50, 964)
target_y_train_select: (50, 2)
source_x_train: (11377, 964)
source_y_train

## Set Cluster 2 as the target domain

In [14]:

target_malware_x, target_malware_y, source_malware_x, source_malware_y =  filter_label(malware_x, malware_y, [2])
print("Malware data ...")
print("Target: {}".format(target_malware_x.shape))
print("Target; {}".format(target_malware_y.shape))
print("Source {}".format(source_malware_x.shape))
print("Source {}".format(source_malware_y.shape))

print("Normal data ...")
print("Target: {}".format(target_normal_x.shape))
print("Target: {}".format(target_normal_y.shape))
print("Source {}".format(source_normal_x.shape))
print("Source {}".format(source_normal_y.shape))

#concatenate source and target
source_malware_y = np.apply_along_axis(change_attack_label, 1, source_malware_y)
target_malware_y = np.apply_along_axis(change_attack_label, 1, target_malware_y)


source_x = np.concatenate((source_malware_x, source_normal_x), axis = 0)
source_y = np.concatenate((source_malware_y, source_normal_y), axis = 0)


target_x = np.concatenate((target_malware_x, target_normal_x), axis = 0)
target_y = np.concatenate((target_malware_y, target_normal_y), axis = 0)


#one-hot encode labels

source_y = tf.keras.utils.to_categorical(source_y, num_classes = 2)
target_y = tf.keras.utils.to_categorical(target_y, num_classes = 2)




print("Combined data ...")
print("Target: {}".format(target_x.shape))
print("Target: {}".format(target_y.shape))
print("Source {}".format(source_x.shape))
print("Source {}".format(source_y.shape))

# Split the data into train and test sets
source_x_train, source_x_test, source_y_train, source_y_test = train_test_split(source_x, source_y, test_size=0.25,random_state=42)
target_x_train, target_x_test, target_y_train, target_y_test = train_test_split(target_x, target_y, test_size=0.5,random_state=42)
        
    

print("train test data ...")
print("Target train: {}".format(target_x_train.shape))
print("Target train: {}".format(target_y_train.shape))
print("Target test: {}".format(target_x_test.shape))
print("Target test: {}".format(target_y_test.shape))
print("Source train: {}".format(source_x_train.shape))
print("Source train: {}".format(source_y_train.shape))
print("Source test: {}".format(source_x_test.shape))
print("Source test: {}".format(source_y_test.shape))
           

Malware data ...
Target: (1438, 964)
Target; (1438, 1)
Source (9270, 964)
Source (9270, 1)
Normal data ...
Target: (7257, 964)
Target: (7257, 1)
Source (7665, 964)
Source (7665, 1)
Combined data ...
Target: (8695, 964)
Target: (8695, 2)
Source (16935, 964)
Source (16935, 2)
train test data ...
Target train: (4347, 964)
Target train: (4347, 2)
Target test: (4348, 964)
Target test: (4348, 2)
Source train: (12701, 964)
Source train: (12701, 2)
Source test: (4234, 964)
Source test: (4234, 2)


In [15]:
samples = [20,50,100,200,300,500]

for size in samples: 
    print("---------------------{} labeled samples per class ---------------------------".format(size))
    # Select a random sample of the target data of different sizes 
    idxs = np.random.permutation(target_x_train.shape[0]) 
    split = int(size)
    idx_sample, _= np.split(idxs, [split])
    target_x_train_select =  target_x_train[idx_sample]
    target_y_train_select =  target_y_train[idx_sample]
    
    print("target_x_train_select: {}".format(target_x_train_select.shape))
    print("target_y_train_select: {}".format(target_y_train_select.shape))
    print('source_x_train: {0}'.format(source_x_train.shape))
    print('source_y_train: {0}'.format(source_y_train.shape))


    # We have limited the number of training steps to 2000 to minimize training time.
    # You can change the epochs to a lower number (lowest 1) just to test if the code is functional.
    dann_nn = DANN_NN(source_x_train, source_y_train, 
                  target_x_train_select,  target_y_train_select , 
                  source_x_test, source_y_test, 
                  target_x_test,target_y_test, nSteps=2000)
    

    dann_nn.train()

    
    

---------------------20 labeled samples per class ---------------------------
target_x_train_select: (20, 964)
target_y_train_select: (20, 2)
source_x_train: (12701, 964)
source_y_train: (12701, 2)

== Build Discriminator...

== Build Generator...
====Loss Weights====
g_loss_weight: 0.1
c_loss_weight: 1
136/136 [==============================] - 0s 787us/step
Step 0 ==> G_Loss: 1.3817579746246338 C_Loss: 0.7591164708137512 D_Loss: 1.3920801877975464  
Acc Source: 0.5462919225318847 Acc Target: 0.17456301747930084 f1 Target: 0.29419862340216324 

136/136 [==============================] - 0s 779us/step
Step 1000 ==> G_Loss: 1.9603867530822754 C_Loss: 0.0013383555924519897 D_Loss: 1.161668300628662  
Acc Source: 0.9898441190363723 Acc Target: 0.9176632934682613 f1 Target: 0.8069039913700108 

Training ended
---------------------50 labeled samples per class ---------------------------
target_x_train_select: (50, 964)
target_y_train_select: (50, 2)
source_x_train: (12701, 964)
source_y_tra